In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn import metrics

from warnings import filterwarnings
filterwarnings('ignore')

# Pre-Processing

In [17]:
df_train = pd.read_excel('SP_Train.xlsx')
df_train['Item_Weight'].fillna(df_train['Item_Weight'].mean(),inplace=True)

df_train.loc[
    (df_train['Outlet_Type'] == 'Grocery Store') & (df_train['Outlet_Size'].isna()), 
    'Outlet_Size'
] = 'Small'

df_train.loc[
    (df_train['Outlet_Type'] == 'Supermarket Type1') & (df_train['Outlet_Size'].isna()), 
    'Outlet_Size'
] = 'Small'

df_train.loc[
    (df_train['Outlet_Type'] == 'Supermarket Type2') & (df_train['Outlet_Size'].isna()), 
    'Outlet_Size'
] = 'Medium'

df_train.loc[
    (df_train['Outlet_Type'] == 'Supermarket Type3') & (df_train['Outlet_Size'].isna()), 
    'Outlet_Size'
] = 'Medium'

df_train['Item_Fat_Content'].replace({'LF':'Low Fat','reg':'Regular','low fat':'Low Fat'},inplace=True)

In [18]:
# Encode categorical variables
train_data_encoded = pd.get_dummies(df_train, drop_first=True)

# Separate features and target
X = train_data_encoded.drop(columns=['Item_Outlet_Sales'])
y = train_data_encoded['Item_Outlet_Sales']


In [6]:
#sepertate features and target

Features = np.array(X)
Target = y.values
Target = np.reshape(Target, (-1, 1))

In [7]:
#splittting data into training and testing data
Features_train,Features_test,Target_train,Target_test = train_test_split(Features,Target,test_size=0.2,random_state=42)

# Built-in Model

In [8]:

#build model with GradientBoostingRegressor
model = GradientBoostingRegressor()

#fit the model
model.fit(Features_train,Target_train)

#predict the model
Target_pred = model.predict(Features_test)

#calculate R-Squared
r2_score = metrics.r2_score(Target_test,Target_pred)
print("R-Squared:",r2_score)

#calculate Mean Absolute Error
mae = metrics.mean_absolute_error(Target_test,Target_pred)
print("Mean Absolute Error:",mae)

#calculate Mean Squared Error
mse = metrics.mean_squared_error(Target_test,Target_pred)
print("Mean Squared Error:",mse)

R-Squared: 0.5968219817382175
Mean Absolute Error: 773.683700560104
Mean Squared Error: 1193661.173240373


In [9]:
# Get hyperparameters (parameters used to initialize the model)
hyperparameters = model.get_params()
print("Hyperparameters:", hyperparameters)


Hyperparameters: {'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}


In [21]:
class Node():
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, var_red=None, value=None):
        ''' constructor ''' 
        
        # for decision node
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.var_red = var_red
        
        # for leaf node
        self.value = value

In [22]:
class DecisionTreeRegressor():
    def __init__(self, min_samples_split=2, max_depth=2):
        self.root = None
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):
      X, Y = dataset[:, :-1], dataset[:, -1]
      num_samples, num_features = np.shape(X)
      best_split = {}
      if num_samples >= self.min_samples_split and curr_depth <= self.max_depth:
          feature_indices = np.array(range(num_features))
          best_split = self.get_best_split(dataset, num_samples, feature_indices)
          if best_split["var_red"] > 0:  # Proceed only if a valid split is found
              left_subtree = self.build_tree(best_split["dataset_left"], curr_depth + 1)
              right_subtree = self.build_tree(best_split["dataset_right"], curr_depth + 1)
              return Node(
                  feature_index=best_split["feature_index"],
                  threshold=best_split["threshold"],
                  left=left_subtree,
                  right=right_subtree
              )
      # Create a leaf node if no valid split is found
      leaf_value = self.calculate_leaf_value(Y)
      return Node(value=leaf_value)

    def get_best_split(self, dataset, num_samples, feature_indices):
        def evaluate_split(feature_index):
            best_split_local = {
                "feature_index": None,
                "threshold": None,
                "dataset_left": None,
                "dataset_right": None,
                "var_red": -float("inf")
            }
            max_var_red_local = -float("inf")
            feature_values = dataset[:, feature_index]
            possible_thresholds = np.unique(feature_values)
            for threshold in possible_thresholds:
                dataset_left, dataset_right = self.split(dataset, feature_index, threshold)
                if len(dataset_left) > 0 and len(dataset_right) > 0:
                    y, left_y, right_y = dataset[:, -1], dataset_left[:, -1], dataset_right[:, -1]
                    curr_var_red = self.variance_reduction(y, left_y, right_y)
                    if curr_var_red > max_var_red_local:
                        best_split_local = {
                            "feature_index": feature_index,
                            "threshold": threshold,
                            "dataset_left": dataset_left,
                            "dataset_right": dataset_right,
                            "var_red": curr_var_red
                        }
                        max_var_red_local = curr_var_red
            return best_split_local

        # Parallelize across features
        splits = Parallel(n_jobs=-1, verbose=1)(delayed(evaluate_split)(feature_index) for feature_index in feature_indices)

        # Find the overall best split
        best_split = max(splits, key=lambda x: x["var_red"])
        return best_split


    def split(self, dataset, feature_index, threshold):
        ''' function to split the data '''

        dataset_left = np.array([row for row in dataset if row[feature_index]<=threshold])
        dataset_right = np.array([row for row in dataset if row[feature_index]>threshold])
        return dataset_left, dataset_right

    def variance_reduction(self, parent, l_child, r_child):
        if len(l_child) == 0 or len(r_child) == 0:
            return 0  # No variance reduction if one of the children is empty
        weight_l = len(l_child) / len(parent)
        weight_r = len(r_child) / len(parent)
        reduction = np.var(parent) - (weight_l * np.var(l_child) + weight_r * np.var(r_child))
        return reduction


    def calculate_leaf_value(self, Y):
        ''' function to compute leaf node '''

        val = np.mean(Y)
        return val

    def print_tree(self, tree=None, indent=" "):
        ''' function to print the tree '''

        if not tree:
            tree = self.root

        if tree.value is not None:
            print(tree.value)

        else:
            print("X_"+str(tree.feature_index), "<=", tree.threshold, "?", tree.var_red)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.left, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.right, indent + indent)

    def fit(self, X, Y):
        ''' function to train the tree '''

        dataset = np.concatenate((X, Y.reshape(-1, 1)), axis=1)
        self.root = self.build_tree(dataset)

    def make_prediction(self, x, tree):
        ''' function to predict new dataset '''

        if tree.value!=None: return tree.value
        feature_val = x[tree.feature_index]
        if feature_val<=tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)

    def predict(self, X):
        ''' function to predict a single data point '''

        preditions = [self.make_prediction(x, self.root) for x in X]
        return preditions

In [23]:
from tqdm.notebook import tqdm

In [24]:
class GradientBoosting:
  def __init__(self, learning_rate = 0.1, n_trees = 100, min_samples = 2, max_depth = 3):
    self.learning_rate = learning_rate
    self.n_trees = n_trees
    self.min_samples = min_samples
    self.max_depth = max_depth
    self.trees = []
    self.mean_predict = 0

  def fit(self, X, y):
    X = np.array(X)
    y = np.array(y)

    self.trees = []
    self.mean_predict = np.mean(y, axis=0)
    y_pred = np.full(X.shape[0], self.mean_predict)

    for _ in range(self.n_trees):
        residual = y - np.array(y_pred)
        tree = DecisionTreeRegressor(
            min_samples_split=self.min_samples,
            max_depth=self.max_depth
        )
        tree.fit(X, residual)
        self.trees.append(tree)
        y_pred += self.learning_rate * np.array(tree.predict(X)).flatten()


  def predict(self, X):
    y_pred = np.full(X.shape[0], self.mean_predict)
    for tree in self.trees:
      y_pred += self.learning_rate * tree.predict(X).flatten()
    return y_pred

In [25]:
model = GradientBoosting(learning_rate=0.1, n_trees=1, min_samples=2, max_depth=3)
model.fit(Features_train, Target_train)

# Make predictions on the test set
Target_pred = model.predict(Features_test)

# Calculate Mean Squared Error (MSE)
mse = metrics.mean_squared_error(Target_test,Target_pred)
print("Mean Squared Error:", mse)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 6398 and the array at index 1 has size 40934404

In [ ]:
Target_pred